# 01 FTH

            First notebook in the workflow. It loads the raw holograms, centers them,
            optionally applies the Ewald projection, defines an ROI, creates a simple
            radially symmetric Butterworth smooth beamstop mask, reconstructs the FTH
            image, and saves the nested `data` dictionary to HDF5.

            Output: `processed/Logs/data_recon_ImId_<im_id>_<user>.hdf5`.

In [1]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt


def fit_horizontal_band(
    pattern,
    nedge=10,
    band_center=None,
    band_width=None,
    band_edge=None,
    band_amplitude=None,
    bump_center=None,
    bump_sigma=None,
    plot=True,
):
    """
    Fit a horizontal detector-band artifact using the left/right edges
    of a 2D scattering pattern.

    Model
    -----
    measured profile =
        polynomial background
        + Gaussian bump
        + negative smoothed-box artifact

    Parameters
    ----------
    pattern : 2D ndarray
        Scattering pattern.

    nedge : int
        Number of columns averaged on each side.

    band_center : float, optional
        Initial estimate of band center [pixels].
        Default = image center.

    band_width : float, optional
        Initial estimate of band width [pixels].

    band_edge : float, optional
        Initial estimate of edge smoothing [pixels].

    band_amplitude : float, optional
        Initial estimate of the depth of the negative band.

    bump_center : float, optional
        Initial estimate of the Gaussian bump center [pixels].

    bump_sigma : float, optional
        Initial estimate of Gaussian sigma [pixels].

    plot : bool
        If True, show diagnostic plots.

    Returns
    -------
    band_2d : ndarray
        Estimated NEGATIVE horizontal-band artifact,
        same shape as pattern.

        Correct with:

            corrected = pattern - band_2d

    gaussian_2d : ndarray
        Fitted Gaussian contribution to the natural profile,
        repeated horizontally to match pattern.shape.

    fit_info : dict
        Fitted parameters and 1D fit components.
    """

    pattern = np.asarray(pattern, dtype=float)

    if pattern.ndim != 2:
        raise ValueError("pattern must be a 2D array.")

    ny, nx = pattern.shape
    y = np.arange(ny, dtype=float)

    # =========================================================
    # Get vertical profiles from left/right detector edges
    # =========================================================

    left_profile = np.nanmean(pattern[:, :nedge], axis=1)
    right_profile = np.nanmean(pattern[:, -nedge:], axis=1)

    profile = 0.5 * (left_profile + right_profile)

    # =========================================================
    # Model components
    # =========================================================

    def smooth_box(y, amplitude, center, width, edge_sigma):
        """
        Positive smooth box made from two error-function edges.
        """

        y1 = center - width / 2
        y2 = center + width / 2

        return amplitude / 2 * (
            erf((y - y1) / (np.sqrt(2) * edge_sigma))
            - erf((y - y2) / (np.sqrt(2) * edge_sigma))
        )

    def polynomial_background(y, c0, c1, c2):
        """
        Slowly varying polynomial background.
        """

        x = y - ny / 2

        return c0 + c1 * x + c2 * x**2

    def gaussian_bump(y, amplitude, center, sigma):
        """
        Broad Gaussian scattering contribution.
        """

        return amplitude * np.exp(
            -0.5 * ((y - center) / sigma)**2
        )

    def model(
        y,
        c0,
        c1,
        c2,
        bump_amp,
        bump_center_fit,
        bump_sigma_fit,
        band_amp,
        band_center_fit,
        band_width_fit,
        band_edge_fit,
    ):

        polynomial = polynomial_background(
            y,
            c0,
            c1,
            c2,
        )

        gaussian = gaussian_bump(
            y,
            bump_amp,
            bump_center_fit,
            bump_sigma_fit,
        )

        band = smooth_box(
            y,
            band_amp,
            band_center_fit,
            band_width_fit,
            band_edge_fit,
        )

        return polynomial + gaussian - band

    # =========================================================
    # Initial guesses
    # =========================================================

    if band_center is None:
        band_center = ny / 2

    if band_width is None:
        band_width = ny * 0.08

    if band_edge is None:
        band_edge = max(2, band_width / 10)

    if bump_center is None:
        bump_center = ny / 2

    if bump_sigma is None:
        bump_sigma = ny / 3

    baseline = np.nanmedian(profile)

    bump_amplitude = max(
        np.nanmax(profile) - baseline,
        1e-12
    )

    # ---------------------------------------------------------
    # Estimate initial band depth if not supplied
    # ---------------------------------------------------------

    if band_amplitude is None:

        distance = np.abs(y - band_center)

        inside = distance < band_width / 2

        outside = (
            (distance > band_width)
            & (distance < 2 * band_width)
        )

        if np.any(inside) and np.any(outside):

            band_amplitude = (
                np.nanmedian(profile[outside])
                - np.nanmedian(profile[inside])
            )

        else:

            band_amplitude = 0.05 * (
                np.nanmax(profile) - np.nanmin(profile)
            )

        band_amplitude = max(
            band_amplitude,
            1e-12
        )

    p0 = [
        baseline,          # c0
        0.0,               # c1
        0.0,               # c2

        bump_amplitude,
        bump_center,
        bump_sigma,

        band_amplitude,
        band_center,
        band_width,
        band_edge,
    ]

    # =========================================================
    # Fit bounds
    # =========================================================

    lower = [
        -np.inf,       # c0
        -np.inf,       # c1
        -np.inf,       # c2

        0,             # bump amplitude
        0,             # bump center
        1,             # bump sigma

        0,             # band amplitude
        0,             # band center
        1,             # band width
        0.2,           # band edge
    ]

    upper = [
        np.inf,
        np.inf,
        np.inf,

        np.inf,
        ny,
        2 * ny,

        np.inf,
        ny,
        ny,
        ny / 2,
    ]

    # =========================================================
    # Fit
    # =========================================================

    valid = np.isfinite(profile)

    popt, pcov = curve_fit(
        model,
        y[valid],
        profile[valid],
        p0=p0,
        bounds=(lower, upper),
        maxfev=50000,
    )

    # =========================================================
    # Separate fitted components
    # =========================================================

    fitted_polynomial = polynomial_background(
        y,
        *popt[:3]
    )

    fitted_gaussian = gaussian_bump(
        y,
        *popt[3:6]
    )

    fitted_band_positive = smooth_box(
        y,
        *popt[6:]
    )

    # Actual detector artifact is negative
    band_1d = -fitted_band_positive

    # Natural profile without artifact
    fitted_background = (
        fitted_polynomial
        + fitted_gaussian
    )

    # Complete measured-profile model
    fitted_total = (
        fitted_polynomial
        + fitted_gaussian
        + band_1d
    )

    # =========================================================
    # Convert 1D components to 2D
    # =========================================================

    band_2d = np.repeat(
        band_1d[:, None],
        nx,
        axis=1
    )

    gaussian_2d = np.repeat(
        fitted_gaussian[:, None],
        nx,
        axis=1
    )

    # =========================================================
    # Diagnostic plots
    # =========================================================

    if plot:

        # -----------------------------------------------------
        # Complete fit
        # -----------------------------------------------------

        plt.figure(figsize=(10, 5))

        plt.plot(
            y,
            left_profile,
            alpha=0.4,
            label="Left edge"
        )

        plt.plot(
            y,
            right_profile,
            alpha=0.4,
            label="Right edge"
        )

        plt.plot(
            y,
            profile,
            linewidth=1.5,
            label="Average edge profile"
        )

        plt.plot(
            y,
            fitted_total,
            linewidth=2.5,
            label="Complete fit"
        )

        plt.plot(
            y,
            fitted_background,
            "--",
            linewidth=2,
            label="Natural profile"
        )

        plt.plot(
            y,
            fitted_polynomial,
            "--",
            alpha=0.7,
            label="Polynomial background"
        )

        plt.plot(
            y,
            fitted_polynomial + fitted_gaussian,
            linewidth=1.5,
            label="Polynomial + Gaussian"
        )

        plt.fill_between(
            y,
            fitted_background,
            fitted_total,
            alpha=0.2,
            label="Band artifact"
        )

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Horizontal-band fit")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Individual extracted components
        # -----------------------------------------------------

        plt.figure(figsize=(10, 4))

        plt.plot(
            y,
            fitted_gaussian,
            label="Gaussian bump"
        )

        plt.plot(
            y,
            band_1d,
            label="Band artifact"
        )

        plt.axhline(
            0,
            linewidth=1
        )

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Extracted fit components")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Parameters
        # -----------------------------------------------------

        print("\nFitted Gaussian:")
        print(f"  amplitude = {popt[3]:.6g}")
        print(f"  center    = {popt[4]:.2f} px")
        print(f"  sigma     = {popt[5]:.2f} px")

        print("\nFitted horizontal band:")
        print(f"  amplitude = {popt[6]:.6g}")
        print(f"  center    = {popt[7]:.2f} px")
        print(f"  width     = {popt[8]:.2f} px")
        print(f"  edge sigma= {popt[9]:.2f} px")

    # =========================================================
    # Output information
    # =========================================================

    names = [
        "background_offset",
        "background_linear",
        "background_quadratic",
        "bump_amplitude",
        "bump_center",
        "bump_sigma",
        "band_amplitude",
        "band_center",
        "band_width",
        "band_edge",
    ]

    fit_info = {
        "parameters": dict(zip(names, popt)),
        "covariance": pcov,

        "left_profile": left_profile,
        "right_profile": right_profile,
        "profile": profile,

        "polynomial": fitted_polynomial,
        "gaussian_1d": fitted_gaussian,
        "background": fitted_background,

        "band_1d": band_1d,

        "fit_profile": fitted_total,
    }

    return band_2d, gaussian_2d, fit_info
    
    

In [2]:
import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.
from mask_store import MaskStore
from data_loading import SextantsNexusLoader, image_ids, load_average

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib qt
try:
    %load_ext jupyter_black
except Exception:
    pass

/usr/lib/python3/dist-packages/pytools/persistent_dict.py:52: RecommendedHashNotFoundWarning: Unable to import recommended hash 'siphash24.siphash13', falling back to 'hashlib.sha256'. Run 'python3 -m pip install siphash24' to install the recommended hash.
  warn("Unable to import recommended hash 'siphash24.siphash13', "


Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
GPU unavailable


## Folders and user

In [3]:
BASEFOLDER = find_basefolder()

# Keep the beamline path for later; the second assignment is active for now.
RAW_FOLDER = "/home/experiences/sextants/com-sextants/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"
RAW_FOLDER = "../COMET_20260902_Cocoons_Laser_raw/"
RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"

RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260526_Test_Cocoons_CDI/"

DATAFOLDER = RAW_FOLDER
RAW_DATA_KIND = "sextants_nexus"  # "existing" or "sextants_nexus"
USER = "rb"

folder_general = helper.create_folder(join(BASEFOLDER, "processed"))
folder_logs = helper.create_folder(join(folder_general, "Logs"))

print("Raw folder:", RAW_FOLDER)
print("Output folder:", folder_general)

Raw folder: /nfs/ruche/sextants-soleil/com-sextants/COMET_20260526_Test_Cocoons_CDI/
Output folder: /home/experiences/sextants/com-sextants/SEXT_NEW/processed


## Define raw images

In [7]:
# Edit these labels and scan IDs for the current dataset.
# The labels become data["holo"][label] groups in the HDF5 file.
# id and dark_id accept one ID or a list; lists are averaged pixel-by-pixel.
hologram_inputs = {
    "+": {
        "id":  [452, 464,466,468,470],
        "dark_id": None,
        "mask_id": None,
        "stitched_file": "processed/stitched/stitched_moved_beamstop_plus_rb.npz",
    },
    "-": {
        "id": [453,465,467,469,471],
        "dark_id": None,
        "mask_id": None,
        "stitched_file": "processed/stitched/stitched_moved_beamstop_minus_rb.npz",
    },
}


pid=[126]
did=128
nid=[127]
#pid+1


hologram_inputs = {
    # mask_id=None auto-detects a saved mask for the frame's own ID.
    # Set another image ID to reuse its compatible raw-coordinate mask.
    # Set stitched_file to a 00b .npz output to use a stitched raw image.
    "+": {
        "id": pid,
        "dark_id": did,
        "mask_id": 95,
        "stitched_file": None,
    },
    "-": {
        "id": nid,
        "dark_id": did,
        "mask_id": 95,
        "stitched_file": None,
    },
}








# Which labels should be used for the FTH difference hologram?
positive_label = "+"
reference_label = "-"

crop = None
project_ewalds_sphere = False
ewald_method = "cubic"
heraldo = False

## Derived paths and experimental setup

In [8]:
# Everything in this cell is derived from the folders and raw-image cells above.
raw_loader = (
    SextantsNexusLoader(RAW_FOLDER) if RAW_DATA_KIND == "sextants_nexus" else None
)
if raw_loader is None:
    raise ValueError(
        "Automatic experimental metadata currently requires RAW_DATA_KIND='sextants_nexus'"
    )

im_ids = image_ids(hologram_inputs[positive_label]["id"])
im_id = int(im_ids[0])  # First ID is used only for metadata and filenames.
positive_file = raw_loader.path_for(im_id)
ccd_dist_source = f"/scan_{im_id:04d}/scan_data/data_03"
with h5py.File(positive_file, "r") as handle:
    if ccd_dist_source not in handle:
        raise KeyError(f"Missing {ccd_dist_source} in {positive_file}")
    ccd_dataset = handle[ccd_dist_source]
    ccd_dist_mm = float(np.asarray(ccd_dataset[()]).squeeze())
ccd_dist_m = (700.0 - ccd_dist_mm) / 1000.0

setup_frame = raw_loader.load(im_id)
if "energy_eV" not in setup_frame.metadata:
    raise KeyError(f"No photon energy found in positive image {im_id} NeXus metadata")

energy = float(setup_frame.metadata["energy_eV"])
experimental_setup = {
    "ccd_dist": ccd_dist_m,
    "ccd_dist_source": ccd_dist_source,
    "ccd_dist_image_id": im_id,
    "px_size": 11.0e-6,
    "binning": 1,
    "oversaturation": 60e3,
    "energy": energy,
    "lambda": helper.photon_energy_wavelength(energy, input_unit="eV"),
}
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)
mnemonics = loading.mnemonics
DATA_H5 = join(folder_logs, f"data_recon_ImId_{im_id:04d}_{USER}.hdf5")

data = {
    "workflow": "FTH_phase_retrieval_4_notebook_sequence",
    "user": USER,
    "data_file": DATA_H5,
    "experimental_setup": experimental_setup,
    "holo": hologram_inputs,
    "hologram_labels": list(hologram_inputs.keys()),
    "positive_label": positive_label,
    "reference_label": reference_label,
    "heraldo": heraldo,
    "crop": crop,
}

print("Positive images:", im_ids)
print("Output HDF5:", DATA_H5)
print(
    "CCD distance:",
    experimental_setup["ccd_dist"],
    "m from",
    experimental_setup["ccd_dist_source"],
)
print("Pixel size:", experimental_setup["px_size"], "m")
print("Energy:", experimental_setup["energy"], "eV")

Positive images: [126]
Output HDF5: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/Logs/data_recon_ImId_0126_rb.hdf5
CCD distance: 0.587066 m from /scan_0126/scan_data/data_03
Pixel size: 1.1e-05 m
Energy: 784.6712586012738 eV


## Load raw data

In [9]:
mask_store = MaskStore(join(BASEFOLDER, "processed", "mask_pixels"))
for label, state in data["holo"].items():
    print(f"Loading {label}: image {state['id']}")
    stitched_file = state.get("stitched_file")
    stitched_mask = None
    if stitched_file:
        stitched_path = (
            stitched_file
            if os.path.isabs(stitched_file)
            else join(BASEFOLDER, stitched_file)
        )
        with np.load(stitched_path, allow_pickle=False) as stitched:
            image = np.asarray(stitched["image"], dtype=float)
            stitched_mask = np.asarray(stitched["mask_pixel"], dtype=np.uint8)
            state["raw_metadata"] = {"energy_eV": float(stitched["energy_eV"])}
            state["exposure"] = float(stitched["reference_exposure"])
        spe_name = stitched_path
        print(f"  stitched input: {stitched_path}")
    else:
        if raw_loader is None:
            image, spe_name = wf.load_processing(DATAFOLDER, state["id"], crop=crop)
            state["raw_metadata"] = {}
        else:
            frame = load_average(raw_loader, state["id"])
            image, spe_name = frame.image, str(frame.source)
            state["raw_metadata"] = dict(frame.metadata)
            state["exposure"] = frame.exposure
    if state.get("dark_id") is not None and not stitched_file:
        if raw_loader is None:
            dark, _ = wf.load_processing(DATAFOLDER, int(state["dark_id"]), crop=crop)
        else:
            dark = load_average(raw_loader, state["dark_id"]).image

        x = dark[100:200, 100:200].ravel()
        y = image[100:200, 100:200].ravel()
        slope, offset = np.polyfit(x, y, 1)
        fig,ax=plt.subplots()
        ax.scatter(x,y)
        ax.plot(x, slope*x+offset)
        corrected_dark = slope * dark + offset
        image = image - corrected_dark
    state["image"] = image
    state["spe_name"] = spe_name
    requested_mask_id = state.get("mask_id")
    mask_id = requested_mask_id
    first_image_id = image_ids(state["id"])[0]
    if mask_id is None and mask_store.exists(first_image_id):
        mask_id = first_image_id
    if stitched_mask is not None:
        state["mask_pixel_raw"] = stitched_mask
        state["mask_id_used"] = "stitched"
    elif mask_id is None:
        state["mask_pixel_raw"] = np.zeros(image.shape, dtype=np.uint8)
        state["mask_id_used"] = None
        print("  no saved user mask; using all detector pixels")
    else:
        if not mask_store.exists(mask_id):
            raise FileNotFoundError(f"Requested mask_id={mask_id} does not exist")
        state["mask_pixel_raw"] = mask_store.load(mask_id, image.shape)
        state["mask_id_used"] = mask_id
        print(
            f"  raw mask from image {mask_id}: {state['mask_pixel_raw'].sum()} pixels"
        )
    if state["mask_pixel_raw"].shape != image.shape:
        raise ValueError(
            f"Mask shape {state['mask_pixel_raw'].shape} != image shape {image.shape}"
        )

plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image"] for label in plot_labels]))

Loading +: image [126]
  raw mask from image 95: 25766 pixels
Loading -: image [127]
  raw mask from image 95: 25766 pixels
Plot order: ['+', '-']


interactive(children=(FloatRangeSlider(value=(-2032.3786994056547, 4429695.067285411), description='contrast',…

interactive(children=(IntSlider(value=0, description='nr', max=1), Output()), _dom_classes=('widget-interact',…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

## Choose center

In [10]:
cimshow(dark)

interactive(children=(FloatRangeSlider(value=(782793.212, 957547.0310000144), description='contrast', layout=L…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [14]:
plt.close("all")

# Center calibrated from image 13 (the '+' acquisition).
pol = reference_label
c0, c1 = [1002, 1037]
#c0, c1 = [1011,1046]

ic = interactive.InteractiveCenter(data["holo"][pol]["image"], c0=c0, c1=c1)

interactive(children=(FloatRangeSlider(value=(-2029.7530135222205, 2367398.129287103), description='contrast',…

interactive(children=(IntText(value=1002, description='c0 (vert)', step=0), IntText(value=1037, description='c…

In [15]:
data["holo"]["+"]["image"]=np.nan_to_num(data["holo"]["+"]["image"])

data["holo"]["-"]["image"]=np.nan_to_num(data["holo"]["-"]["image"])

In [16]:
# Get center positions from the widget.
center = [ic.c0, ic.c1]
print("Center:", center)
data["center"] = center
data = wf.define_centered_holograms(
    data,
    cci,
    PhR=PhR,
    project_ewalds_sphere=project_ewalds_sphere,
    ewald_method=ewald_method,
)
for label, state in data["holo"].items():
    state["mask_pixel_c"] = (
        wf.center_image(state["mask_pixel_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
# A difference hologram is valid only where both input images are valid.
mask_pixel = np.maximum(
    data["holo"][positive_label]["mask_pixel_c"],
    data["holo"][reference_label]["mask_pixel_c"],
).astype(np.uint8)
data["mask_pixel_raw_by_label"] = {
    label: state["mask_pixel_raw"] for label, state in data["holo"].items()
}
data["mask_pixel"] = mask_pixel
plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image_c"] for label in plot_labels]))

Center: [1001, 1011]
Plot order: ['+', '-']


interactive(children=(FloatRangeSlider(value=(-2012.43791486009, 4429695.067285413), description='contrast', l…

interactive(children=(IntSlider(value=0, description='nr', max=1), Output()), _dom_classes=('widget-interact',…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

## Smooth Butterworth mask and FTH reconstruction

In [17]:
# Smooth masks are used only for FTH; data['mask_pixel'] remains binary for phase retrieval.
# mask_pixel was loaded in raw coordinates and centered after center selection.
butterworth_radius = 35
butterworth_order = 4
mask_pixel_smoothing_pixels = 3
prop_dist = 0
phase = 0

shape = data["holo"][positive_label]["image_c"].shape
if mask_pixel.shape != shape:
    raise ValueError(
        f"Centered mask shape {mask_pixel.shape} != hologram shape {shape}"
    )
mask_beamstop_smooth_recipe = {
    "type": "radial_butterworth_disk",
    "radius": butterworth_radius,
    "order": butterworth_order,
}
mask_beamstop_smooth = wf.butterworth_disk_mask(
    shape, butterworth_radius, butterworth_order
)
mask_pixel_fth_recipe = {
    "type": "binary_dilation_gaussian",
    "dilation_pixels": mask_pixel_smoothing_pixels,
    "sigma": mask_pixel_smoothing_pixels,
}
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float), mask_pixel_smoothing_pixels, mask_pixel_smoothing_pixels
)
mask_multiplier = (1 - mask_beamstop_smooth) * (1 - mask_pixel_fth)

pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float)
ref = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
factor, offset = cci.dyn_factor(
    pos * (1 - mask_pixel),
    ref * (1 - mask_pixel),
    method="correlation",
    verbose=True,
    plot=True,
)

factor=1
offset=0
pos_scaled = pos / factor
holo_unmasked = pos_scaled - ref - offset
holo_masked = holo_unmasked * mask_multiplier
recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

data["mask_pixel"] = mask_pixel
data["mask_beamstop_smooth_recipe"] = mask_beamstop_smooth_recipe
data["mask_pixel_fth_recipe"] = mask_pixel_fth_recipe
data["factor"] = factor
data["offset"] = offset
data["holo"][positive_label]["image_c_norm"] = wf.normalize_image(pos_scaled)
data["holo"][reference_label]["image_c_norm"] = wf.normalize_image(ref)
data["focus_fth"] = {
    "prop_dist": prop_dist,
    "prop_dist_unit": "um",
    "phase": phase,
    "dx": 0,
    "dy": 0,
    "roi": None,
    "operation": "-",
}
data["fth_recipe"] = {
    "positive_label": positive_label,
    "reference_label": reference_label,
    "center": data["center"],
    "mask_beamstop_smooth_recipe": mask_beamstop_smooth_recipe,
    "mask_pixel_fth_recipe": mask_pixel_fth_recipe,
    "project_ewalds_sphere": data["project_ewalds_sphere"],
    "ewald_method": ewald_method,
    "contrast": "positive / factor - reference - offset",
}

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
tmp = holo_masked
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.1, 99.9))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].add_patch(
    plt.Circle(
        (shape[1] / 2, shape[0] / 2),
        butterworth_radius,
        fill=False,
        edgecolor="red",
        linewidth=1.5,
    )
)
ax[0].set_title("mask * hologram")

tmp = np.real(recon_unmasked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("FTH before masking")
tmp = np.real(recon_masked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[2].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[2].set_title("FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

Linear Fit: 3.4871*x + -15606.4979


In [18]:
plt.close("all")

cimshow(np.clip(pos+ref, -1000, 1000))

interactive(children=(FloatRangeSlider(value=(-1000.0, 1000.0), description='contrast', layout=Layout(width='5…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

## Define ROI from masked FTH reconstruction

In [19]:
# Preview the masked FTH reconstruction. The next cell applies the calibrated fixed ROI.
# fig, ax = cimshow(np.real(recon_masked))

In [20]:
# Fixed ROI calibrated with the image-13 center above.
roi = [597, 785, 526, 719]
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print("ROI:", roi)

ROI: [597, 785, 526, 719]


In [21]:
plt.close("all")
roicrop=np.s_[900:-900,900:-900]

In [22]:

def butterworth_radial(radius, order, shape):
    ny, nx = shape
    y, x = np.indices((ny, nx))
    cy = (ny - 1) / 2
    cx = (nx - 1) / 2
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    filt = 1 / (1 + (r / radius)**(2 * order))
    return filt
bwfilt=1-butterworth_radial(radius=100, order=4, shape=pos.shape)

## Focus FTH reconstruction

In [23]:
#plt.close("all")
roi_s2=np.s_[:,:]
crop=0
if crop==0:
    roicrop=np.s_[:,:]
else:
    roicrop=np.s_[crop:-crop,crop:-crop]

if False:
    band, gaussian,fit = fit_horizontal_band(
        pos,
        nedge=100,
        band_center=1024,
        band_width=80,
        band_edge=11,
    )
    pos=pos-gaussian-band
    
    band, gaussian,fit = fit_horizontal_band(
        ref,
        nedge=100,
        band_center=1024,
        band_width=80,
        band_edge=11,
    )
    ref=ref-gaussian-band

In [26]:
factor, offset = cci.dyn_factor(
    pos * (1 - mask_pixel),
    ref * (1 - mask_pixel),
    method="correlation",
    verbose=True,
    plot=True,
)
factor=1
offset=0
pos_scaled = pos / factor-offset
holo_unmasked = pos_scaled - ref 
bwfilt=1-butterworth_radial(radius=150, order=5, shape=pos.shape)
holo_masked = holo_unmasked * (1+0*mask_multiplier)*bwfilt
cimshow(holo_masked)

Linear Fit: 3.4871*x + -15606.4979


interactive(children=(FloatRangeSlider(value=(-58504.89869125763, 523051.4439231888), description='contrast', …

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [29]:
plt.close("all")
# Use the focusCDI widget to tune propagation distance and phase.
# The selected values are committed to the HDF5 data dictionary in the next cell.
# Manual starting values: propagation is in micrometres, phase in radians,
# and dx/dy are sub-pixel shifts. Edit these before creating the sliders.
prop_dist, phase, dx, dy = -13.85, 1.94, 0, 0.0
prop_dist, phase, dx, dy = -15.83, 2.01, 0, 0.0

roi_s2=np.s_[400:650,475:675]
prop_dist, phase, dx, dy = -15.83, 2.01, 0, 0.2
focus_operation = "-"
focus_sliders = rec.focusCDI(
    1*((pos_scaled)* mask_multiplier*bwfilt)[roicrop],
    1*((ref)* mask_multiplier*bwfilt)[roicrop],
    roi_s2,
    mask=1,
    phase=phase,
    prop_dist=prop_dist,
    dx=dx,
    dy=dy,
    experimental_setup=data["experimental_setup"],
    operation=focus_operation,
    max_prop_dist=30,
    scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

interactive(children=(FloatSlider(value=-15.83, description='propagation[um]', layout=Layout(width='90%'), max…

In [273]:
# Commit selected focus values and recompute the saved FTH reconstruction.
prop_dist = slider_prop.value
phase = slider_phase.value
dx = slider_dx.value
dy = slider_dy.value

focus_fth = {
    "prop_dist": prop_dist,
    "prop_dist_unit": "um",
    "phase": phase,
    "dx": dx,
    "dy": dy,
    "roi": roi,
    "operation": focus_operation,
}
data["focus_fth"] = focus_fth

recon_unmasked = wf.fth_reconstruct(
    holo_unmasked,
    experimental_setup,
    fth,
    prop_dist=prop_dist,
    phase=phase,
    dx=dx,
    dy=dy,
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase, dx=dx, dy=dy
)

data["recon"] = recon_masked[roi_s]
data["recon_description"] = (
    "Complex FTH reconstruction after propagation, phase shift, Butterworth masking, and ROI crop."
)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
tmp = np.real(recon_unmasked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].set_title("Focused FTH before masking")
tmp = np.real(recon_masked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("Focused FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

print("prop_dist:", prop_dist)
print("phase:", phase)

im_ids = image_ids(data["holo"][positive_label]["id"])
im_id = int(im_ids[0])
png_title = f"Focused FTH after masking - im_id {im_id}"
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(png_title)
ax.set_axis_off()
plt.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.show()

data["fth_png"] = png_name
print("Saved PNG:", png_name)

prop_dist: -15.28
phase: 0.603407346410207
Saved PNG: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/FTH_recon_ImId_0478_rb.png


## Save

In [274]:
# Always write the display PNG in the same final cell as the HDF5 result.
im_ids = image_ids(data["holo"][positive_label]["id"])
im_id = int(im_ids[0])
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"Focused FTH after masking - im_id {im_id}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)
data["fth_png"] = png_name
written = wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", written)
print("Saved PNG:", png_name)

Saved HDF5: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/Logs/data_recon_ImId_0478_rb.hdf5
Saved PNG: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/FTH_recon_ImId_0478_rb.png


In [275]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_ids:", _summary_im)
print("topo_ids:", _summary_topo)
print("dark_ids (+):", _summary_holo.get(_summary_pos, {}).get("dark_id"))
print("dark_ids (-):", _summary_holo.get(_summary_ref, {}).get("dark_id"))
print("HDF5:", _summary_h5)

im_ids: [478]
topo_ids: [479]
dark_ids (+): 480
dark_ids (-): 480
HDF5: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/Logs/data_recon_ImId_0478_rb.hdf5


In [276]:
plt.close("all")

In [251]:
# Acquisition ID summary
_id_holo = data.get("holo", {})
print("im_ids:", {label: state.get("id") for label, state in _id_holo.items()})
print("dark_ids:", {label: state.get("dark_id") for label, state in _id_holo.items()})

im_ids: {'+': [481], '-': [482]}
dark_ids: {'+': 483, '-': 483}
